In [ ]:
# 05. Autoencoder for Fraud Detection

# This notebook implements:
# 1. Semi-supervised Autoencoder (train only on normal samples)
# 2. Supervised Autoencoder (train on full labeled data)
# 3. Comparison of both training regimes
# 4. Cost-Sensitive Evaluation (FN=$500, FP=$2)
# 5. Error Analysis (FP/FN inspection)
# 6. Latent Space Visualization

import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score, roc_auc_score, 
    precision_recall_curve, precision_score, 
    recall_score, f1_score
)

try:
    current_dir = Path.cwd()
    if (current_dir / 'src').exists():
        PROJECT_ROOT = current_dir
    elif (current_dir.parent / 'src').exists():
        PROJECT_ROOT = current_dir.parent
    else:
        PROJECT_ROOT = Path.cwd().parent
except Exception:
    PROJECT_ROOT = Path.cwd()

print(f"Project root: {PROJECT_ROOT}")

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_processed_data(dataset='ieee'):
    """Load pre-processed data from processed folder"""
    data_dir = PROJECT_ROOT / 'data' / 'processed'
    
    train = pd.read_parquet(data_dir / f'{dataset}_train.parquet')
    test = pd.read_parquet(data_dir / f'{dataset}_test.parquet')
    
    with open(data_dir / 'dataset_info.json', 'r') as f:
        info = json.load(f)
    
    raw_cols = info[dataset]['raw_features']
    eng_cols = info[dataset]['engineered_features']
    all_cols = raw_cols + eng_cols
    
    return train, test, raw_cols, eng_cols, all_cols

# Load data
train_ieee, test_ieee, raw_cols, eng_cols, all_cols = load_processed_data('ieee')

print(f"Raw features: {len(raw_cols)}")
print(f"Engineered features: {len(eng_cols)}")
print(f"Total features: {len(all_cols)}")
print(f"\nTrain shape: {train_ieee.shape}")
print(f"Test shape: {test_ieee.shape}")
print(f"Train fraud rate: {train_ieee['isFraud'].mean():.4f}")
print(f"Test fraud rate: {test_ieee['isFraud'].mean():.4f}")

# 1. Feature Preparation

X_train_raw = train_ieee[all_cols].values
X_test_raw = test_ieee[all_cols].values
y_train = train_ieee['isFraud'].values
y_test = test_ieee['isFraud'].values

print(f"\nX_train shape: {X_train_raw.shape}")
print(f"X_test shape: {X_test_raw.shape}")

# Handle missing values
print(f"\nTrain missing values: {np.isnan(X_train_raw).sum()}")
print(f"Test missing values: {np.isnan(X_test_raw).sum()}")

if np.isnan(X_train_raw).sum() > 0:
    print("Filling missing values...")
    for i in range(X_train_raw.shape[1]):
        col_median = np.nanmedian(X_train_raw[:, i])
        X_train_raw[np.isnan(X_train_raw[:, i]), i] = col_median
        X_test_raw[np.isnan(X_test_raw[:, i]), i] = col_median

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"\nX_train shape after scaling: {X_train.shape}")
print(f"X_test shape after scaling: {X_test.shape}")




In [ ]:
# 2. Autoencoder Model

class Autoencoder(nn.Module):
    """Autoencoder for anomaly detection"""
    def __init__(self, input_dim, encoding_dim=32, hidden_dims=[128, 64]):
        super(Autoencoder, self).__init__()
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.append(nn.Linear(prev_dim, h_dim))
            encoder_layers.append(nn.BatchNorm1d(h_dim))
            encoder_layers.append(nn.ReLU())
            encoder_layers.append(nn.Dropout(0.2))
            prev_dim = h_dim
        encoder_layers.append(nn.Linear(prev_dim, encoding_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder
        decoder_layers = []
        prev_dim = encoding_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.append(nn.Linear(prev_dim, h_dim))
            decoder_layers.append(nn.BatchNorm1d(h_dim))
            decoder_layers.append(nn.ReLU())
            decoder_layers.append(nn.Dropout(0.2))
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def get_reconstruction_error(self, x):
        """Compute reconstruction error (MSE) for each sample"""
        with torch.no_grad():
            reconstructed = self.forward(x)
            mse = ((x - reconstructed) ** 2).mean(dim=1)
        return mse.cpu().numpy()
    
    def get_latent(self, x):
        """Get latent representation (for visualization)"""
        with torch.no_grad():
            return self.encoder(x).cpu().numpy()


class EnsembleAutoencoder:
    """Ensemble Autoencoder with multiple encoding dimensions"""
    def __init__(self, input_dim, encoding_dims=[8, 16, 32, 64]):
        self.input_dim = input_dim
        self.encoding_dims = encoding_dims
        self.models = []
    
    def _train_single(self, X_train, encoding_dim, epochs=100, batch_size=4096, verbose=True):
        """Train a single autoencoder"""
        val_size = int(len(X_train) * 0.1)
        X_train_sub = torch.FloatTensor(X_train[:-val_size]).to(device)
        X_val = torch.FloatTensor(X_train[-val_size:]).to(device)
        
        train_dataset = TensorDataset(X_train_sub, X_train_sub)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        model = Autoencoder(self.input_dim, encoding_dim=encoding_dim).to(device)
        
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
        criterion = nn.MSELoss()
        
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0
            for batch_X, _ in train_loader:
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_X)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * batch_X.size(0)
            
            epoch_loss /= len(X_train_sub)
            
            model.eval()
            with torch.no_grad():
                val_outputs = model(X_val)
                val_loss = criterion(val_outputs, X_val).item()
            
            scheduler.step(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_state = model.state_dict().copy()
            else:
                patience_counter += 1
            
            if patience_counter >= 15:
                break
            
            if verbose and (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        model.load_state_dict(best_state)
        return model
    
    def fit(self, X_train, y_train, mode='semi_supervised', epochs=100, batch_size=4096, verbose=True):
        """Train all autoencoders in ensemble"""
        if mode == 'semi_supervised':
            normal_idx = (y_train == 0)
            X_train_selected = X_train[normal_idx]
            if verbose:
                print(f"Semi-supervised mode: {len(X_train_selected):,} normal samples")
        else:
            X_train_selected = X_train
            if verbose:
                print(f"Supervised mode: {len(X_train_selected):,} all samples")
        
        self.models = []
        for i, enc_dim in enumerate(self.encoding_dims):
            if verbose:
                print(f"\nTraining encoding dimension {enc_dim} ({i+1}/{len(self.encoding_dims)})")
            model = self._train_single(X_train_selected, enc_dim, epochs, batch_size, verbose)
            self.models.append(model)
        
        return self
    
    def predict_anomaly_scores(self, X):
        """Predict anomaly scores (average reconstruction error across ensemble)"""
        X_tensor = torch.FloatTensor(X).to(device)
        all_scores = []
        
        for model in self.models:
            model.eval()
            scores = model.get_reconstruction_error(X_tensor)
            all_scores.append(scores)
        
        return np.mean(all_scores, axis=0)
    
    def get_latent_representations(self, X, model_idx=0):
        """Get latent representations from a specific model"""
        X_tensor = torch.FloatTensor(X).to(device)
        self.models[model_idx].eval()
        return self.models[model_idx].get_latent(X_tensor)


In [ ]:

# 3. Semi-supervised Autoencoder

input_dim = X_train.shape[1]
print(f"\nInput dimension: {input_dim}")

print("\n" + "="*60)
print("Training: Semi-supervised Ensemble Autoencoder")
print("="*60)

ensemble_semi = EnsembleAutoencoder(input_dim, encoding_dims=[8, 16, 32, 64])
ensemble_semi.fit(X_train, y_train, mode='semi_supervised', epochs=100, batch_size=4096, verbose=True)

test_scores_semi = ensemble_semi.predict_anomaly_scores(X_test)

auprc_semi = average_precision_score(y_test, test_scores_semi)
roc_auc_semi = roc_auc_score(y_test, test_scores_semi)

print(f"\nTest Results (Semi-supervised Ensemble Autoencoder):")
print(f"  AUPRC: {auprc_semi:.4f}")
print(f"  ROC AUC: {roc_auc_semi:.4f}")

# 4. Train Supervised Autoencoder

print("\n" + "="*60)
print("Training: Supervised Ensemble Autoencoder")
print("="*60)

ensemble_sup = EnsembleAutoencoder(input_dim, encoding_dims=[8, 16, 32, 64])
ensemble_sup.fit(X_train, y_train, mode='supervised', epochs=100, batch_size=4096, verbose=True)

test_scores_sup = ensemble_sup.predict_anomaly_scores(X_test)

auprc_sup = average_precision_score(y_test, test_scores_sup)
roc_auc_sup = roc_auc_score(y_test, test_scores_sup)

print(f"\nTest Results (Supervised Ensemble Autoencoder):")
print(f"  AUPRC: {auprc_sup:.4f}")
print(f"  ROC AUC: {roc_auc_sup:.4f}")


In [ ]:

# 5. Cost-Sensitive Evaluation

def normalize_scores(scores):
    """Normalize scores to [0, 1] range"""
    scores_min = scores.min()
    scores_max = scores.max()
    if scores_max - scores_min > 0:
        return (scores - scores_min) / (scores_max - scores_min)
    return scores

test_scores_semi_norm = normalize_scores(test_scores_semi)
test_scores_sup_norm = normalize_scores(test_scores_sup)

def find_cost_optimal_threshold(y_true, y_scores, fn_cost=500, fp_cost=2):
    """Find threshold that minimizes total cost"""
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    
    costs = []
    f1_scores = []
    
    for t in thresholds:
        y_pred = (y_scores >= t).astype(int)
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        
        total_cost = fn * fn_cost + fp * fp_cost
        costs.append(total_cost)
        
        p = tp / (tp + fp + 1e-9)
        r = tp / (tp + fn + 1e-9)
        f1_scores.append(2 * p * r / (p + r + 1e-9) if (p + r) > 0 else 0)
    
    best_idx = np.argmin(costs)
    best_threshold = thresholds[best_idx]
    min_cost = costs[best_idx]
    
    f1_best_idx = np.argmax(f1_scores)
    f1_optimal_threshold = thresholds[f1_best_idx]
    
    return best_threshold, min_cost, f1_optimal_threshold, thresholds, costs, f1_scores

# Compute cost-optimal thresholds
cost_threshold_semi, min_cost_semi, f1_threshold_semi, thresholds_semi, costs_semi, f1_scores_semi = find_cost_optimal_threshold(
    y_test, test_scores_semi_norm, fn_cost=500, fp_cost=2
)

cost_threshold_sup, min_cost_sup, f1_threshold_sup, thresholds_sup, costs_sup, f1_scores_sup = find_cost_optimal_threshold(
    y_test, test_scores_sup_norm, fn_cost=500, fp_cost=2
)

print("\n" + "="*60)
print("Cost-Sensitive Evaluation (FN=$500, FP=$2)")
print("="*60)
print(f"\nSemi-supervised Autoencoder:")
print(f"  Cost-optimal threshold: {cost_threshold_semi:.4f}")
print(f"  F1-optimal threshold: {f1_threshold_semi:.4f}")
print(f"  Minimum total cost: ${min_cost_semi:,.2f}")
print(f"  Difference (Cost vs F1): {abs(cost_threshold_semi - f1_threshold_semi):.4f}")

print(f"\nSupervised Autoencoder:")
print(f"  Cost-optimal threshold: {cost_threshold_sup:.4f}")
print(f"  F1-optimal threshold: {f1_threshold_sup:.4f}")
print(f"  Minimum total cost: ${min_cost_sup:,.2f}")
print(f"  Difference (Cost vs F1): {abs(cost_threshold_sup - f1_threshold_sup):.4f}")

# 6. Evaluation at Cost-Optimal Threshold

y_pred_semi = (test_scores_semi_norm > cost_threshold_semi).astype(int)
y_pred_sup = (test_scores_sup_norm > cost_threshold_sup).astype(int)

print("\n" + "="*60)
print(f"Evaluation at Cost-Optimal Threshold")
print("="*60)

print(f"\nSemi-supervised Autoencoder (threshold={cost_threshold_semi:.4f}):")
print(f"  AUPRC: {auprc_semi:.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_semi):.4f}")
print(f"  Recall: {recall_score(y_test, y_pred_semi):.4f}")
print(f"  F1: {f1_score(y_test, y_pred_semi):.4f}")

print(f"\nSupervised Autoencoder (threshold={cost_threshold_sup:.4f}):")
print(f"  AUPRC: {auprc_sup:.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_sup):.4f}")
print(f"  Recall: {recall_score(y_test, y_pred_sup):.4f}")
print(f"  F1: {f1_score(y_test, y_pred_sup):.4f}")

In [ ]:

# 7. Error Analysis

print("\n" + "="*60)
print("Error Analysis - Supervised Autoencoder")
print("="*60)

fp_idx = np.where((y_test == 0) & (y_pred_sup == 1))[0]
fn_idx = np.where((y_test == 1) & (y_pred_sup == 0))[0]
tp_idx = np.where((y_test == 1) & (y_pred_sup == 1))[0]

print(f"False Positives (Normal flagged as Fraud): {len(fp_idx)}")
print(f"False Negatives (Fraud missed): {len(fn_idx)}")
print(f"True Positives (Fraud caught): {len(tp_idx)}")

# Sample analysis of False Positives
print("\n--- False Positives Analysis (First 10 samples) ---")
if len(fp_idx) > 0:
    sample_fp = test_ieee.iloc[fp_idx[:min(10, len(fp_idx))]]
    if 'TransactionAmt' in sample_fp.columns:
        print(f"Transaction amounts: min=${sample_fp['TransactionAmt'].min():.2f}, "
              f"max=${sample_fp['TransactionAmt'].max():.2f}, "
              f"mean=${sample_fp['TransactionAmt'].mean():.2f}")
    if 'hour' in sample_fp.columns:
        print(f"Hour distribution: {dict(sample_fp['hour'].value_counts().sort_index().head(5))}")

# Sample analysis of False Negatives
print("\n--- False Negatives Analysis (First 10 samples) ---")
if len(fn_idx) > 0:
    sample_fn = test_ieee.iloc[fn_idx[:min(10, len(fn_idx))]]
    if 'TransactionAmt' in sample_fn.columns:
        print(f"Transaction amounts: min=${sample_fn['TransactionAmt'].min():.2f}, "
              f"max=${sample_fn['TransactionAmt'].max():.2f}, "
              f"mean=${sample_fn['TransactionAmt'].mean():.2f}")

# 8. Latent Space Visualization

print("\n" + "="*60)
print("Latent Space Visualization")
print("="*60)

# Sample test data for visualization
sample_size = 2000
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), sample_size, replace=False)
X_sample = X_test[sample_idx]
y_sample = y_test[sample_idx]

# Get latent representations from supervised autoencoder
latent_sup = ensemble_sup.get_latent_representations(X_sample, model_idx=0)

# 2D visualization using PCA
pca = PCA(n_components=2)
latent_2d = pca.fit_transform(latent_sup)

# 9. Visualization

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 9.1 PR Curves
precision_semi, recall_semi, _ = precision_recall_curve(y_test, test_scores_semi_norm)
precision_sup, recall_sup, _ = precision_recall_curve(y_test, test_scores_sup_norm)

axes[0, 0].plot(recall_semi, precision_semi, linewidth=2, 
                label=f'Semi-supervised (AUPRC={auprc_semi:.4f})')
axes[0, 0].plot(recall_sup, precision_sup, linewidth=2,
                label=f'Supervised (AUPRC={auprc_sup:.4f})')
axes[0, 0].axhline(y=0.6132, color='red', linestyle='--', linewidth=2,
                   label=f'LightGBM Baseline (AUPRC=0.6132)')
axes[0, 0].axhline(y=y_test.mean(), color='gray', linestyle=':', linewidth=1.5,
                   label=f'Random (AUPRC={y_test.mean():.4f})')
axes[0, 0].set_xlabel('Recall')
axes[0, 0].set_ylabel('Precision')
axes[0, 0].set_title('Precision-Recall Curve')
axes[0, 0].legend(loc='upper right')
axes[0, 0].grid(True, alpha=0.3)

# 9.2 Cost Curve - Semi-supervised
axes[0, 1].plot(thresholds_semi, costs_semi, 'b-', linewidth=2)
axes[0, 1].axvline(x=cost_threshold_semi, color='red', linestyle='--', 
                   label=f'Cost-Optimal: {cost_threshold_semi:.4f}')
axes[0, 1].axvline(x=f1_threshold_semi, color='green', linestyle='--', 
                   label=f'F1-Optimal: {f1_threshold_semi:.4f}')
axes[0, 1].set_xlabel('Threshold')
axes[0, 1].set_ylabel('Total Cost ($)')
axes[0, 1].set_title('Semi-supervised - Cost Analysis')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 9.3 Cost Curve - Supervised
axes[0, 2].plot(thresholds_sup, costs_sup, 'b-', linewidth=2)
axes[0, 2].axvline(x=cost_threshold_sup, color='red', linestyle='--', 
                   label=f'Cost-Optimal: {cost_threshold_sup:.4f}')
axes[0, 2].axvline(x=f1_threshold_sup, color='green', linestyle='--', 
                   label=f'F1-Optimal: {f1_threshold_sup:.4f}')
axes[0, 2].set_xlabel('Threshold')
axes[0, 2].set_ylabel('Total Cost ($)')
axes[0, 2].set_title('Supervised - Cost Analysis')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 9.4 Anomaly Score Distribution (Semi-supervised)
axes[1, 0].hist(test_scores_semi[y_test==0], bins=50, alpha=0.6, label='Normal', density=True)
axes[1, 0].hist(test_scores_semi[y_test==1], bins=50, alpha=0.6, label='Fraud', density=True)
threshold_99 = np.percentile(test_scores_semi[y_test==0], 99)
axes[1, 0].axvline(x=threshold_99, color='red', linestyle='--', label=f'99th Percentile')
axes[1, 0].set_xlabel('Reconstruction Error')
axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title('Anomaly Score Distribution (Semi-supervised)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 9.5 Error Analysis - Score Distribution
fp_scores = test_scores_sup_norm[fp_idx] if len(fp_idx) > 0 else []
fn_scores = test_scores_sup_norm[fn_idx] if len(fn_idx) > 0 else []
tp_scores = test_scores_sup_norm[tp_idx] if len(tp_idx) > 0 else []

axes[1, 1].hist(fp_scores, bins=20, alpha=0.5, label=f'FP (n={len(fp_idx)})', color='red')
axes[1, 1].hist(fn_scores, bins=20, alpha=0.5, label=f'FN (n={len(fn_idx)})', color='orange')
axes[1, 1].hist(tp_scores, bins=20, alpha=0.5, label=f'TP (n={len(tp_idx)})', color='green')
axes[1, 1].axvline(x=cost_threshold_sup, color='blue', linestyle='--', 
                   label=f'Threshold={cost_threshold_sup:.4f}')
axes[1, 1].set_xlabel('Anomaly Score')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Score Distribution by Error Type')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 9.6 Latent Space Visualization
colors = ['blue' if y == 0 else 'red' for y in y_sample]
axes[1, 2].scatter(latent_2d[:, 0], latent_2d[:, 1], c=colors, alpha=0.5, s=10)
axes[1, 2].set_xlabel('PC1')
axes[1, 2].set_ylabel('PC2')
axes[1, 2].set_title('Latent Space (PCA projection)')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='blue', alpha=0.5, label='Normal'),
                   Patch(facecolor='red', alpha=0.5, label='Fraud')]
axes[1, 2].legend(handles=legend_elements)
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'data/processed' / 'autoencoder_complete_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

# Results Summary

results_summary = pd.DataFrame({
    'Model': [
        'Semi-supervised Autoencoder',
        'Supervised Autoencoder',
        'Supervised Neural Network',
        'LightGBM Baseline'
    ],
    'AUPRC': [
        auprc_semi,
        auprc_sup,
        0.5157,
        0.6132
    ],
    'ROC AUC': [
        roc_auc_semi,
        roc_auc_sup,
        0.9133,
        0.9431
    ],
    'Cost-Optimal Threshold': [
        f"{cost_threshold_semi:.4f}",
        f"{cost_threshold_sup:.4f}",
        "0.0708",
        "0.0708"
    ],
    'Min Total Cost': [
        f"${min_cost_semi:,.2f}",
        f"${min_cost_sup:,.2f}",
        "$148,464",
        "$148,464"
    ]
})

print("\n" + "="*80)
print("COMPLETE RESULTS SUMMARY - Autoencoder vs Baselines")
print("="*80)
print(results_summary.to_string(index=False))

